In [1]:
import pandas as pd

In [3]:
# 1. Load the datasets
bbc_df = pd.read_excel("bbc_saynis_iyo_caafimaad_scraped.xlsx")
goobjoog_df = pd.read_excel("goobjoog_caafimaad_articles_REPAIRED.xlsx")

In [5]:
# 2. Standardize BBC Columns
bbc_df = bbc_df.rename(columns={
    'url': 'url',
    'Headline': 'headline',
    'Body': 'body',
    'category': 'category',
    'source': 'source'
})

In [6]:
# We keep only the columns we want
bbc_df = bbc_df[['url', 'headline', 'body', 'category', 'source']]

In [7]:
# 3. Standardize Goobjoog Columns
# Goobjoog doesn't have 'category' or 'source', so we add them manually
goobjoog_df = goobjoog_df.rename(columns={
    'URL': 'url',
    'Headline': 'headline',
    'Body': 'body'
})

In [8]:
# Adding missing info for Goobjoog
goobjoog_df['category'] = 'Caafimaad'
goobjoog_df['source'] = 'Goobjoog'

In [9]:
# Keep only the relevant columns
goobjoog_df = goobjoog_df[['url', 'headline', 'body', 'category', 'source']]

In [10]:
# 4. Combine the datasets
combined_df = pd.concat([bbc_df, goobjoog_df], ignore_index=True)

In [11]:
# 1. Count the total number of duplicate rows based on the URL
duplicate_count = combined_df.duplicated(subset=['url']).sum()

In [12]:
print(f"Total duplicate URLs found: {duplicate_count}")

Total duplicate URLs found: 0


In [31]:
# Check for missing data or placeholders before removal
missing_data_df = combined_df[
    combined_df['headline'].isna() | 
    combined_df['body'].isna() | 
    (combined_df['headline'].str.contains("No Headline Found", na=False, case=False))
]

In [32]:
print(f"Total rows with missing/placeholder data: {len(missing_data_df)}")
if not missing_data_df.empty:
    print("Preview of missing rows:")
    print(missing_data_df[['url', 'source', 'headline']].head())

Total rows with missing/placeholder data: 0


In [26]:
# Remove rows where Headline or Body is missing (NaN)
combined_df = combined_df.dropna(subset=['headline', 'body'])

In [27]:
# Remove rows where the scraper put the "No Headline Found" placeholder
combined_df = combined_df[~combined_df['headline'].str.contains("No Headline Found", na=False, case=False)]

In [28]:
# Remove rows that have a headline but an empty body string
combined_df = combined_df[combined_df['body'].str.strip() != ""]

In [33]:
print(f"Total rows with missing/placeholder data: {len(missing_data_df)}")


Total rows with missing/placeholder data: 0


In [34]:
# Remove duplicate URLs
initial_count = len(combined_df)
combined_df = combined_df.drop_duplicates(subset=['url'])
final_count = len(combined_df)

In [35]:
print(f"BBC records: {len(bbc_df)}")
print(f"Goobjoog records: {len(goobjoog_df)}")
print(f"Duplicates removed: {initial_count - final_count}")
print(f"Total rows in new dataset: {final_count}")

# 6. Export to the new file
output_file = "standardized_caafimaad_master.xlsx"
combined_df.to_excel(output_file, index=False)

print(f"\n✅ Master file created: {output_file}")

BBC records: 499
Goobjoog records: 658
Duplicates removed: 0
Total rows in new dataset: 1144

✅ Master file created: standardized_caafimaad_master.xlsx
